# Assignment 2 - Metabolic Modeling (Week 2)

## Setup and Imports

In [1]:
%pip install cobra
%pip install panda
%pip install escher



Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Created wheel for panda: filename=panda-0.3.1-py3-none-any.whl size=7288 sha256=dbdfaf30e84b825b747cba72a775fb3a65155ecb871ec2b5b84c9bd8146f88bc
  Stored in directory: c:\users\ruben\appdata\local\pip\cache\wheels\b3\62\ac\41e27178e8597be2da908681ca1e75e66255a488040cca1638
Successfully built panda

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.5 MB 2.5 MB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.5 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.7 MB/s  0:00:00
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 3.3 MB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 3.6 MB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 3.5 MB/s  0:00:00
   ---------------------------------------- 0.0/5.5 MB ? eta -:--:--
   ----- ---------------------------------- 0.8/5.5 MB 4.4 MB/s eta 0:00:02
   ----------- ---------------------------- 1.6/5.5 MB 3.8 MB/s eta 0:00:02
   ----------------- ---------------------- 2.4/5.5 MB 3.6 MB/s eta 0:00:01
   ---------------------- ----------------- 3.1/5.5 MB 3.7 MB/s eta 0:00:01
   ---------------------------- -------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

In [1]:
import cobra as cobra
import pandas as pd 
import escher



In [3]:
data = pd.read_csv("KEN3170_Assignment_2026_e_coli_core_expression.csv")
reaction_data = dict(zip(data["# Reaction ID"], data[" reaction activity [mmol/gDW/h] "]))
builder = escher.Builder(map_name="e_coli_core.Core metabolism", eaction_data=reaction_data)
builder
# you still need to load the data by yourself

Builder()

In [4]:
# after observing we can notice that "GAPD", "PGK", "PGM", "ENO", "PYK" form a linear pathway
linear_pathway = ["GAPD", "PGK", "PGM", "ENO", "PYK"]
# only include this path
comparison = data[data["# Reaction ID"].isin(linear_pathway)].copy()
# order such that the path has the correct values
comparison["# Reaction ID"] = pd.Categorical(comparison["# Reaction ID"],categories=linear_pathway,ordered=True)
comparison = comparison.sort_values("# Reaction ID")
comparison


,# Reaction ID,reaction activity [mmol/gDW/h]
46,GAPD,24.5
3,PGK,24.0
7,PGM,21.7
30,ENO,29.3
22,PYK,28.2


In [5]:
from cobra import Reaction
reaction_data["PGK"]

24.0

In [6]:
# get the activity values
activities = comparison[" reaction activity [mmol/gDW/h] "]
print("Are all activities equal?", activities.nunique() == 1)


Are all activities equal? False


### 1.a
As shown above, the **maximal reaction activities differ within the linear pathway**. This is due to the fact that they represent **capacity, but not the flux itself**. Therefore, unlike the steady-state fluxes observed in the interactive session, they are **not required to be equal due to mass balance constraints**.

### 1.b
The grey arrows show either **zero or no data (`nd`)**. Zero represents that the **maximal reaction activity for this reaction is zero**, whereas no data means that it **has not yet been determined in this model (missing information)**.


In [7]:
model = cobra.io.load_json_model('e_coli_core.json')
model.reactions.get_by_id("PFK")
# the model already defined weather or not a reaction is reversible based on the lower and upper bounds

Reaction identifier,PFK
Name,Phosphofructokinase
Memory address,0x1f29b3f6270
Stoichiometry,"atp_c + f6p_c --> adp_c + fdp_c + h_c ATP C10H12N5O13P3 + D-Fructose 6-phosphate --> ADP C10H12N5O10P2 + D-Fructose 1,6-bisphosphate + H+"
GPR,b3916 or b1723
Lower bound,0.0
Upper bound,1000.0


# Task 2

In [ ]:
def modify_bounds(reaction_data,model):


    # leave the lower flux as it is for ATPM
    atpm_lower_bound = model.reactions.get_by_id("ATPM").lower_bound

    for reaction in model.reactions:
        # special case
        if reaction.id == "EX_glc__D_e":
            reaction.lower_bound = -1000.0
            reaction.upper_bound = 1000.0
            continue

        # get all of the reactions that are both in reaction data and model
        if reaction.id in reaction_data:
            value = reaction_data[reaction.id]
            # print(value)
            # reversible reactions
            if reaction.reversibility: #originally reversible
                reaction.lower_bound = - value
                reaction.upper_bound = value
            else: #originally NOT reversible
                reaction.upper_bound = value

    # does high absolute default bound means bounds of lb= -1000 and ub=1000
    # if yes then change - model.reactions.get_by_id("EX_glc__D_e").lower_bound = -1000

    model.reactions.get_by_id("ATPM").lower_bound = atpm_lower_bound
    
    return model
    


In [11]:
model = modify_bounds(reaction_data, model)
bounds_df = pd.DataFrame([{"Reaction": reaction.id,"Lower Bound": reaction.lower_bound,
"Upper Bound": reaction.upper_bound}for reaction in model.reactions])

pd.set_option('display.max_rows', None)
bounds_df

,Reaction,Lower Bound,Upper Bound
0,PFK,0.00,13.10
1,PFL,0.00,0.00
2,PGI,-11.10,11.10
3,PGK,-24.00,24.00
4,PGL,0.00,7.30
5,ACALD,-0.00,0.00
6,AKGt2r,-0.00,0.00
7,PGM,-21.70,21.70
8,PIt2r,-5.20,5.20
9,ALCD2x,-0.00,0.00
